# Project Brief 

In this project, I want to predict the total number of bike rentals in one hour (cnt).
The dataset contains hourly data from 2011 to 2012.
Each row represents one hour.

The data includes information like hour of the day, season, weekday, temperature, humidity, windspeed, and weather condition.

My goal is to build a regression model that can estimate how many bikes will be rented in a given hour using this information.

This problem is useful because it can help manage bike availability and avoid shortages or too many unused bikes.

I will first build simple baseline models, then try more advanced models.

I will measure performance using MAE and RMSE.

A good result means my model performs better than the baseline and works well on future data.

## 5 Features I Expect to Matter

1. hr (hour of the day)
I expect this to matter because people rent bikes more during rush hours like morning and evening.

2. temp (temperature)
When the weather is warm, more people are likely to rent bikes. Very cold weather may reduce rentals.

3. weathersit (weather condition)
Bad weather like rain or snow will probably reduce the number of bike rentals.

4. workingday
On working days, people may use bikes to go to work, so rentals might be higher than weekends.

5. season
Different seasons may affect bike usage. For example, winter might have fewer rentals than summer.

## Questions
#### 1. What is a prediction error that would be acceptable in the real world here?

An acceptable error would be small enough that the bike stations can still manage supply correctly. For example, being wrong by 10–20 bikes per hour might be acceptable, but being wrong by 100 bikes would cause problems. The acceptable error also depends on how many bikes are usually rented in that hour.

### 2. What would make your model unusable or misleading?

The model would be unusable if it makes very large errors during hours when many bikes are renteds. It would also be misleading if it performs well on training data but fails on future/unseen data.

# Load data and column understanding

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns


In [ ]:
df = pd.read_csv('C:\\Users\\Mr.Zabit\\Documents\\ml_projects\\bike_rental_project\\data\\raw\\hour.csv')
df.head(3)



In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

### Questions 
• Which columns are categorical vs numeric?

• Which columns are IDs or dates? Which should (or should not) be used directly?

| Column     | Meaning                                       | Type        | Use as Predictor? |
| ---------- | --------------------------------------------- | ----------- | ----------------- |
| instant    | Row ID                                        | ID          | No                |
| dteday     | Date of record                                | date        | No                |
| season     | Season (1=Spring, 2=Summer, 3=Fall, 4=Winter) | categorical | Yes               |
| yr         | Year (0=2011, 1=2012)                         | categorical | Yes               |
| mnth       | Month (1–12)                                  | categorical | Yes               |
| hr         | Hour of day (0–23)                            | categorical | Yes               |
| holiday    | Whether it is a holiday (0/1)                 | categorical | Yes               |
| weekday    | Day of week (0=Sunday … 6=Saturday)           | categorical | Yes               |
| workingday | Whether it is a working day (0/1)             | categorical | Yes               |
| weathersit | Weather situation (1–4)                       | categorical | Yes               |
| temp       | Normalized temperature                        | numeric     | Yes               |
| atemp      | Normalized “feels like” temperature           | numeric     | Yes               |
| hum        | Humidity                                      | numeric     | Yes               |
| windspeed  | Windspeed                                     | numeric     | Yes               |
| casual     | Casual rentals                                | numeric     | No                |
| registered | Registered rentals                            | numeric     | No                |
| cnt        | Total rentals (target)                        | numeric     | Target            |




# Exploratory Data Analysis(EDA) 


### Hypothesis:

I expect cnt to be right-skewed — most hours have moderate rentals, few hours have very high rentals.

In [ ]:
sns.histplot(df['cnt'], bins=50, kde=True)
plt.title('Distribution of Bike Rentals (cnt)')
plt.xlabel('Total Rentals')
plt.ylabel('Frequency')
plt.show()


### Hypothesis

Rush hours (7–9 AM, 17–19 PM) → higher rentals

Weekends → lower rentals for commuting

Bad weather → fewer rentals

Winter → fewer rentals than summer

In [ ]:
cols = ['hr', 'weekday', 'workingday', 'season', 'weathersit']
n_cols = 3  # number of plots per row
n_rows = (len(cols) + n_cols - 1) // n_cols  # compute number of rows

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*4))
axes = axes.flatten()  # flatten to 1D array for easy iteration

for i, col in enumerate(cols):
    df.groupby(col)['cnt'].mean().plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Mean Rentals by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Mean cnt')

# Remove any empty subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Hypothesis

Higher temperature → more rentals

High humidity → fewer rentals

In [ ]:
cols = ["atemp", "hum"]
n_cols = 2
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, n_rows*4))
axes = axes.flatten()

for i, col in enumerate(cols):
    df[col + '_bin'] = pd.cut(df[col], bins=10)
    df.groupby(col + '_bin')['cnt'].mean().plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Mean Rentals by {col} Bins')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Mean cnt')

plt.tight_layout()
plt.show()


### 1What 3 patterns are strongest?

Rush hour effect (hr)
The strongest pattern is that rentals increase during morning (7–9) and evening (17–19) hours.

Working day vs non-working day
Rentals are higher on working days compared to non-working days. This suggests many people use bikes for going to work.

Weather / Temperature effect
Rentals increase when temperature is moderate or warm, and decrease during bad weather conditions.

### Do you see any interactions?

Yes, I see some interactions.

One interaction is between hour and workingday. During the working day the rental increases compared non-workingdays, this shows cnt depends on workingdays.